# FLUX Phase 5: Causal Geometry Nodes (CGN)
## Complete Pipeline — Train, Test, Demo, Upload

> **Goal:** Build the CGN layer that replaces neurons with geometry-aware nodes that store causal relationships, not just mappings. Every conclusion stores WHY it was reached.

### What this notebook does:
1. **Setup** — Clone/pull repo, install deps, detect hardware
2. **Load Phases 1–4** — Verify checkpoint chain
3. **Smoke Test** — Verify CGN components build and run
4. **Train** — Causal graph formation, multi-timescale processing, geometry validation, full pipeline
5. **Upload** — Checkpoint → HuggingFace Hub
6. **Test 1** — Causal trace accuracy (every output has traceable chain)
7. **Test 2** — Multi-timescale separation (fast < 5 steps, slow > 10 steps)
8. **Test 3** — Geometry computation correctness (signal bending, mass amplification)
9. **Demo 1** — Trace why a conclusion was reached (penguin/bird reasoning)
10. **Demo 2** — Fast vs slow node activation curves
11. **Interactive** — Build custom causal graphs, process signals, explore
12. **Results** — View RESULTS_PHASE_5.md + full log
13. **Final** — Upload logs → HuggingFace & push to GitHub
14. **Artifacts** — Save checkpoint, logs, results to Kaggle output

### Key Physics:
- `curvature = computation` — signals are bent by manifold geometry, not multiplied by weights
- `causal arrows` — every conclusion stores WHY (source → conclusion with evidence weight)
- `multi-timescale` — fast nodes (syntax, 0.01τ) + medium (semantics, 0.1τ) + slow (concepts, 1.0τ)
- `invalidation propagation` — disprove a cause → automatically weaken all dependent conclusions
- `mass = evidence` — more evidence = stronger pull, no evidence = node fades

### Secrets Required:
- **`HF_TOKEN`** — HuggingFace write token (Add via Kaggle → Add-ons → Secrets)

---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import sys
import subprocess
import importlib
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

# ─────────────────────────────────────────────
# 1. Clone or Pull — always get the absolute
#    latest code from GitHub.
# ─────────────────────────────────────────────
if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("ℹ Repo already exists — pulling latest changes...")

    subprocess.run(
        ['git', 'remote', 'set-url', 'origin', REPO_URL],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )

    subprocess.run(['git', 'checkout', '.'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)
    subprocess.run(['git', 'clean', '-fd'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)

    fetch = subprocess.run(['git', 'fetch', '--all'], cwd=str(WORK_DIR),
                           capture_output=True, text=True)
    if fetch.returncode != 0:
        print(f"  ⚠ git fetch failed: {fetch.stderr.strip()}")
    result = subprocess.run(
        ['git', 'reset', '--hard', 'origin/main'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(result.stdout.strip() or result.stderr.strip())

    head = subprocess.run(
        ['git', 'log', '--oneline', '-3'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(f"\n  Latest commits:\n{head.stdout.strip()}")
    print("\n✓ Pulled & reset to latest origin/main")
else:
    if WORK_DIR.exists():
        import shutil
        shutil.rmtree(str(WORK_DIR))
    print(f"ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")

# ─────────────────────────────────────────────
# 2. Flush cached Python modules so the kernel
#    picks up the freshly-pulled code, not stale
#    imports from a previous cell run.
# ─────────────────────────────────────────────
_stale = [m for m in sys.modules if any(
    m.startswith(p) for p in (
        'cgn', 'manifold', 'causal_graph', 'multi_timescale',
        'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
        'cse', 'field', 'flux_utils', 'wave_types', 'interference',
        'attractor', 'field_ops', 'train_', 'demo_', 'test_',
        'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
    )
)]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"  ✓ Flushed {len(_stale)} cached modules: {_stale[:5]}{'...' if len(_stale) > 5 else ''}")

# ─────────────────────────────────────────────
# 3. Delete stale Phase 5 checkpoint so training
#    runs fresh with the updated code.
# ─────────────────────────────────────────────
_stale_ckpt = WORK_DIR / 'checkpoints' / 'phase5.phase.pt'
if _stale_ckpt.exists():
    _stale_ckpt.unlink()
    print(f"  ✓ Deleted stale checkpoint: {_stale_ckpt.name}")

subprocess.run(['python', 'setup.py'], cwd=str(WORK_DIR),
               capture_output=True, text=True)

# Quick sanity check: verify Phase 5 modules are present
_cgn_path = WORK_DIR / 'phases' / 'phase5' / 'cgn.py'
_cgn_text = _cgn_path.read_text()
assert 'CausalGeometryNode' in _cgn_text, "ERROR: cgn.py missing CausalGeometryNode!"
assert 'forward_with_trace' in _cgn_text, "ERROR: cgn.py missing forward_with_trace!"
print(f"  ✓ cgn.py verified: CausalGeometryNode + forward_with_trace present")

_mt_path = WORK_DIR / 'phases' / 'phase5' / 'multi_timescale.py'
_mt_text = _mt_path.read_text()
assert 'MultiTimescaleCoordinator' in _mt_text, "ERROR: multi_timescale.py missing MultiTimescaleCoordinator!"
print(f"  ✓ multi_timescale.py verified: MultiTimescaleCoordinator present")

print("✓ setup.py refreshed")

---
## Cell 2: Install Dependencies & Setup

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub matplotlib
!python setup.py

---
## Cell 3: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys, torch, platform, importlib
from pathlib import Path

# ── Add project paths ──
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase1'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase2'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase3'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase4'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase5'))

# ── Force-reload all project modules ──
for _mod_name in list(sys.modules.keys()):
    if any(_mod_name.startswith(p) for p in (
        'flux_utils', 'cgn', 'manifold', 'causal_graph', 'multi_timescale',
        'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
        'cse', 'field', 'wave_types', 'interference', 'attractor', 'field_ops',
        'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
    )):
        del sys.modules[_mod_name]

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint, verify_checkpoint_chain,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    PhaseResults, checkpoint_exists,
)

# ── Initialize Phase 5 Logger ──
log = PhaseLogger(phase=5)
log.separator("Phase 5: Causal Geometry Nodes")
log.cell_start("Cell 3 — Hardware & Secrets")

# ── Detect hardware ──
DEVICE = get_device()
hw = get_hardware_info()
log.info(f"Device: {DEVICE}")
log.info(f"PyTorch: {torch.__version__}")
log.info(f"Platform: {hw['platform']}")
for k, v in hw.items():
    print(f"  {k}: {v}")

if torch.cuda.is_available():
    log.info(f"GPU: {hw.get('gpu', 'N/A')}")
    log.info(f"VRAM: {hw.get('gpu_memory', 'N/A')}")
    log.info(f"CUDA: {hw.get('cuda', 'N/A')}")

# ── Load HuggingFace token ──
HF_TOKEN = _resolve_hf_token()
if HF_TOKEN:
    log.success("HuggingFace token loaded")
    print("  ✓ HF token loaded")
else:
    log.warning("No HuggingFace token — checkpoint upload will be skipped")
    print("  ⚠ No HF token — add HF_TOKEN to Kaggle Secrets for auto-upload")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

---
## Cell 4: Load Phases 1–4 Checkpoints + Smoke Test

Verifies the checkpoint chain before Phase 5 builds on top.
- **Phase 1 (CSE):** Encodes text → semantic wave [seq_len, 432]
- **Phase 2 (Field):** Stores patterns as energy attractors in 3D field
- **Phase 4 (TL):** Thermodynamic settling on top of field
- **Phase 5 (CGN):** Will wire causal geometry nodes onto the pipeline

In [ ]:
log.cell_start("Cell 4 — Load Phases 1–4 + Smoke Test")

import torch
import torch.nn.functional as F

# Force-reimport phase modules
import importlib
for _m in ['cse', 'field', 'thermodynamic', 'temperature', 'energy_functions',
           'online_learner', 'cgn', 'manifold', 'causal_graph', 'multi_timescale']:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from cse import ContinuousSemanticEncoder
from field import ResonanceField, FIELD_H, FIELD_W, FIELD_D, FIELD_FEATURES
from thermodynamic import ThermodynamicLearner
from online_learner import OnlineLearner
from cgn import CausalGeometryNode, CGN_CONFIG
from causal_graph import CausalGraph
from multi_timescale import MultiTimescaleCoordinator

# ── Load Phase 1 (CSE) ──
ckpt1 = load_checkpoint(1)
cse = ContinuousSemanticEncoder(**ckpt1.get('config', {}))
cse.load_state_dict(ckpt1['state_dict'])
cse = cse.to(DEVICE).eval()
for p in cse.parameters():
    p.requires_grad = False

# Smoke test CSE
test_wave = cse.encode("smoke test Phase 1")
assert test_wave.full.shape[-1] == 432, f"Bad wave dim: {test_wave.full.shape}"
assert not torch.isnan(test_wave.full).any(), "NaN in wave!"
log.success(f"Phase 1 CSE loaded: {sum(p.numel() for p in cse.parameters()):,} params")
print(f"  ✓ Phase 1 (CSE): wave shape {test_wave.full.shape}")

# ── Load Phase 2 (ResonanceField) ──
ckpt2 = load_checkpoint(2)
field_cfg = ckpt2.get('config', {}).get('field', {})
field = ResonanceField(**field_cfg)
field.load_state_dict(ckpt2['state_dict'])
field = field.to(DEVICE)

# Smoke test Field
vec = test_wave.full.mean(dim=0).to(DEVICE)
field_out, sims, locs = field.query(vec, k=4)
assert not torch.isnan(field_out).any(), "NaN in field query!"
log.success(f"Phase 2 Field loaded: {sum(p.numel() for p in field.parameters()):,} params")
print(f"  ✓ Phase 2 (ResonanceField): field {FIELD_H}×{FIELD_W}×{FIELD_D}×{FIELD_FEATURES}")

# ── Load Phase 4 (ThermodynamicLearner) ──
ckpt4 = load_checkpoint(4)
tl = ThermodynamicLearner(field=field, settle_iterations=10, decay=0.995).to(DEVICE)
tl.load_state(ckpt4['tl_state'])
ol = OnlineLearner(cse=cse, tl=tl, device=DEVICE)
log.success("Phase 4 ThermodynamicLearner loaded")
print(f"  ✓ Phase 4 (TL): loaded from checkpoint")

# ── Smoke test CGN components ──
feature_dim = 512
cgn_test = CausalGeometryNode(0, feature_dim).to(DEVICE)
test_signal = torch.randn(feature_dim, device=DEVICE) * 0.5
cgn_out = cgn_test(test_signal)
assert cgn_out.shape == test_signal.shape, f"CGN output shape mismatch: {cgn_out.shape}"
assert not torch.isnan(cgn_out).any(), "NaN in CGN output!"
log.success(f"CausalGeometryNode smoke test: output shape {cgn_out.shape}")
print(f"  ✓ Phase 5 (CGN): CausalGeometryNode smoke test OK")

# Smoke test MultiTimescaleCoordinator
mtc_test = MultiTimescaleCoordinator(feature_dim).to(DEVICE)
mtc_out = mtc_test(test_signal, steps=2)
assert mtc_out.shape == test_signal.shape
log.success(f"MultiTimescaleCoordinator smoke test: {mtc_test.total_nodes()} nodes, {mtc_test.total_params():,} params")
print(f"  ✓ Phase 5 (MTC): {mtc_test.total_nodes()} nodes, {mtc_test.total_params():,} params")

# Smoke test CausalGraph
cg_test = CausalGraph()
cg_test.add_arrow(0, 1, weight=1.0, reason="test")
trace = cg_test.trace_cause(1, depth=2)
assert len(trace.chain) >= 2
log.success("CausalGraph smoke test: trace works")
print(f"  ✓ Phase 5 (CG): CausalGraph smoke test OK")

del cgn_test, mtc_test, cg_test
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n  Phase 1 model: https://huggingface.co/UnseenGAP/FLUX (phase1.phase.pt)")
print("  Phase 2 model: https://huggingface.co/UnseenGAP/FLUX (phase2.phase.pt)")
print("  Phase 4 model: https://huggingface.co/UnseenGAP/FLUX (phase4.phase.pt)")
log.cell_end("Cell 4 — Load Phases 1–4 + Smoke Test", "PASS")

---
## Cell 5: Training / Load from Checkpoint

> **Smart skip:** If a Phase 5 checkpoint already exists locally or on HuggingFace Hub, this cell loads it and skips training.

**If training from scratch — five stages:**
- **Stage A:** Causal graph formation (build knowledge graph, trace causal chains)
- **Stage B:** Multi-timescale separation measurement (fast vs slow activation)
- **Stage C:** Geometry computation correctness (signal bending, mass amplification)
- **Stage D:** Causal invalidation (disprove cause → weaken conclusions)
- **Stage E:** Full pipeline — CSE → Field → CGN end-to-end

~3 min on GPU • ~15 min on CPU

In [ ]:
log.cell_start("Cell 5 — Training / Load from Checkpoint")

import time
import torch
import torch.nn.functional as F
from cgn import CausalGeometryNode, CGN_CONFIG
from causal_graph import CausalGraph
from multi_timescale import MultiTimescaleCoordinator

# ─────────────────────────────────────────────────────────────────
# SKIP GUARD — if Phase 5 checkpoint already exists, load it
# directly and skip all training stages.
# ─────────────────────────────────────────────────────────────────
_PHASE5_FRESH_TRAIN = not checkpoint_exists(5)

if not _PHASE5_FRESH_TRAIN:
    print("  ✓ Phase 5 checkpoint found — loading saved model (skipping training)")
    log.info("Phase 5 checkpoint exists — loading instead of training")

    ckpt5 = load_checkpoint(5)
    cgn_cfg = ckpt5.get('config', {})
    cgn_network = MultiTimescaleCoordinator(
        feature_dim=cgn_cfg.get('feature_dim', 512),
        n_fast=cgn_cfg.get('n_fast', 16),
        n_medium=cgn_cfg.get('n_medium', 8),
        n_slow=cgn_cfg.get('n_slow', 4),
    ).to(DEVICE)
    cgn_network.load_state(ckpt5['cgn_state'])

    causal_graph = CausalGraph()
    causal_graph.load_state(ckpt5['causal_graph_state'])

    _stored_metrics = ckpt5.get('metrics', {})
    elapsed = _stored_metrics.get('training_time_seconds', 0.0)

    print(f"  ✓ Checkpoint loaded")
    print(f"  ✓ CGN params:        {cgn_network.total_params():,}")
    print(f"  ✓ Causal edges:      {causal_graph.edge_count()}")
    print(f"  ✓ Causal nodes:      {causal_graph.node_count()}")
    print("\n  ⏩ Skipping training — continuing to tests, demos & results")

else:
    # ── FULL TRAINING PATH ────────────────────────────────────────
    start_time = time.time()

    feature_dim = 512
    cgn_network = MultiTimescaleCoordinator(
        feature_dim=feature_dim,
        n_fast=16,
        n_medium=8,
        n_slow=4,
    ).to(DEVICE)
    log.success(f"MultiTimescaleCoordinator built: {cgn_network.total_params():,} params")

    causal_graph = CausalGraph()
    log.success("CausalGraph initialized")

    # ════════════════════════════════════════════
    # Stage A: Causal Graph Formation
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage A: Causal Graph Formation")
    print("=" * 65)

    concept_names = {
        0: "bird", 1: "penguin", 2: "can_fly", 3: "cannot_fly",
        4: "sparrow", 5: "fish", 6: "can_swim", 7: "dolphin",
        8: "mammal", 9: "breathes_air",
    }

    knowledge = [
        (0, 2, 0.8, "birds can fly"),
        (1, 0, 1.0, "penguin is a bird"),
        (1, 3, 0.9, "penguin cannot fly"),
        (4, 0, 1.0, "sparrow is a bird"),
        (4, 2, 0.95, "sparrow can fly"),
        (5, 6, 0.9, "fish can swim"),
        (7, 5, 0.3, "dolphin resembles fish"),
        (7, 8, 1.0, "dolphin is a mammal"),
        (8, 9, 0.95, "mammal breathes air"),
    ]

    for src, tgt, weight, reason in knowledge:
        causal_graph.add_arrow(src, tgt, weight=weight, reason=reason)
        print(f"  ✓ {concept_names[src]:>10} → {concept_names[tgt]:<15}  w={weight:.2f}  {reason}")

    log.metric("causal_edges", causal_graph.edge_count())
    log.metric("causal_nodes", causal_graph.node_count())

    trace_fly = causal_graph.trace_cause(2, depth=3)
    trace_not_fly = causal_graph.trace_cause(3, depth=3)
    print(f"\n  Trace 'can fly':    {[concept_names.get(n, str(n)) for n in trace_fly.chain]}")
    print(f"  Trace 'cannot fly': {[concept_names.get(n, str(n)) for n in trace_not_fly.chain]}")
    print(f"  Conflict detected: {trace_fly.has_conflict or trace_not_fly.has_conflict}")

    # ════════════════════════════════════════════
    # Stage B: Multi-Timescale Separation
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage B: Multi-Timescale Separation")
    print("=" * 65)

    test_signal = torch.randn(feature_dim, device=DEVICE) * 0.5
    cgn_network.reset_states()
    sep = cgn_network.measure_timescale_separation(test_signal, max_steps=100)

    print(f"  Fast nodes activate at step:   {sep['fast_steps_to_activate']}")
    print(f"  Medium nodes activate at step: {sep['medium_steps_to_activate']}")
    print(f"  Slow nodes activate at step:   {sep['slow_steps_to_activate']}")
    log.metric("fast_activation_step", sep['fast_steps_to_activate'])
    log.metric("medium_activation_step", sep['medium_steps_to_activate'])
    log.metric("slow_activation_step", sep['slow_steps_to_activate'])

    # Process real text through CGN
    test_texts = [
        "Birds can fly through the air",
        "Penguins are birds that cannot fly",
        "The quick brown fox jumps over the lazy dog",
    ]
    for text in test_texts:
        with torch.no_grad():
            wave = cse.encode(text)
            wave_vec = wave.full.mean(dim=0).to(DEVICE)
            feature = field.wave_to_feature(wave_vec).detach()
        cgn_network.reset_states()
        result = cgn_network.forward_with_traces(feature, steps=10)
        print(f"  '{text[:50]}' → fast={result.fast_activation:.4f}  med={result.medium_activation:.4f}  slow={result.slow_activation:.4f}")

    # ════════════════════════════════════════════
    # Stage C: Geometry Computation
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage C: Geometry Computation Correctness")
    print("=" * 65)

    node = CausalGeometryNode(0, feature_dim).to(DEVICE)
    signal = torch.randn(feature_dim, device=DEVICE) * 0.5
    output, trace = node.forward_with_trace(signal)

    print(f"  Bending magnitude: {trace.bending_magnitude:.4f}")
    print(f"  Influence strength: {trace.influence_strength:.4f}")

    with torch.no_grad():
        node.mass.copy_(torch.tensor(5.0))
    output_high_mass = node(signal)
    mass_ratio = output_high_mass.norm().item() / max(output.norm().item(), 1e-8)
    print(f"  Mass=5.0 amplification: {mass_ratio:.2f}x")
    log.metric("mass_amplification", f"{mass_ratio:.2f}x")

    # ════════════════════════════════════════════
    # Stage D: Causal Invalidation
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage D: Causal Invalidation")
    print("=" * 65)

    causal_graph.add_arrow(10, 11, weight=1.0, reason="all swans are white")
    causal_graph.add_arrow(11, 12, weight=0.8, reason="therefore this is white")

    summary_before = causal_graph.get_conflict_summary(11)
    print(f"  Before: '{summary_before['conclusion']}' (net={summary_before['net_weight']:.2f})")

    affected = causal_graph.invalidate_cause(10)
    summary_after = causal_graph.get_conflict_summary(11)
    print(f"  After:  '{summary_after['conclusion']}' (net={summary_after['net_weight']:.2f})")
    print(f"  Affected downstream: {affected}")
    log.metric("invalidation_propagation", len(affected))

    # ════════════════════════════════════════════
    # Stage E: Full Pipeline — CSE → Field → CGN
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage E: Full Pipeline — CSE → Field → CGN")
    print("=" * 65)

    pipeline_texts = [
        "The Earth revolves around the Sun",
        "Water freezes at zero degrees Celsius",
        "Gravity pulls objects toward the center of the Earth",
        "Light travels at 300000 kilometers per second",
        "Plants convert sunlight into energy through photosynthesis",
    ]

    for text in pipeline_texts:
        with torch.no_grad():
            wave = cse.encode(text)
            wave_vec = wave.full.mean(dim=0).to(DEVICE)
            feature = field.wave_to_feature(wave_vec).detach()
            cgn_network.reset_states()
            cgn_out = cgn_network(feature, steps=5)

        print(f"  ✓ '{text[:50]}' → cgn_norm={cgn_out.norm().item():.4f}")

    elapsed = time.time() - start_time
    log.metric("training_time", f"{elapsed:.1f}s")
    print(f"\n  ✓ Training complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")

log.cell_end("Cell 5 — Training / Load from Checkpoint", "PASS")

---
## Cell 6: Save Checkpoint & Upload to HuggingFace Hub

> **Smart skip:** If checkpoint already existed, this cell skips save and upload.

In [ ]:
log.cell_start("Cell 6 — Save & Upload Checkpoint")

from datetime import datetime

if not _PHASE5_FRESH_TRAIN:
    ckpt_path = Path('checkpoints/phase5.phase.pt')
    size_mb = ckpt_path.stat().st_size / 1e6 if ckpt_path.exists() else 0
    print(f"  ⏩ Checkpoint already saved — skipping save step")
    print(f"     Local:  {ckpt_path} ({size_mb:.1f} MB)")
    print(f"     Remote: https://huggingface.co/UnseenGAP/FLUX")
    log.info("Checkpoint already existed — save step skipped")
else:
    checkpoint_state = {
        'phase': 5,
        'timestamp': datetime.now().isoformat(),
        'phase1_config': ckpt1.get('config', {}),
        'phase2_config': ckpt2.get('config', {}),
        'tl_state': tl.save_state(),
        'cgn_state': cgn_network.save_state(),
        'causal_graph_state': causal_graph.save_state(),
        'config': {
            'feature_dim': feature_dim,
            'n_fast': 16,
            'n_medium': 8,
            'n_slow': 4,
        },
        'metrics': {
            'causal_edges': causal_graph.edge_count(),
            'causal_nodes': causal_graph.node_count(),
            'total_cgn_params': cgn_network.total_params(),
            'fast_activation_step': sep['fast_steps_to_activate'],
            'slow_activation_step': sep['slow_steps_to_activate'],
            'mass_amplification': mass_ratio,
            'training_time_seconds': elapsed,
        },
    }
    save_checkpoint(5, checkpoint_state)

    ckpt_path = Path('checkpoints/phase5.phase.pt')
    if ckpt_path.exists():
        size_mb = ckpt_path.stat().st_size / 1e6
        log.success(f"Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
        print(f"  ✓ Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
    else:
        log.error("Checkpoint NOT found — save may have failed")

    uploaded = upload_checkpoint_to_hf(phase=5, hf_token=HF_TOKEN)
    if uploaded:
        log.success("Checkpoint uploaded to https://huggingface.co/UnseenGAP/FLUX")
        print("  ✓ Uploaded to HuggingFace Hub")
    else:
        log.warning("Checkpoint upload skipped — check HF_TOKEN")
        print("  ⚠ Upload skipped — no HF token")

    upload_logs_to_hf(phase=5, hf_token=HF_TOKEN)

log.cell_end("Cell 6 — Save & Upload Checkpoint", "PASS")

---
## Cells 7–9: Tests

| Test | Description | Pass Criteria |
|------|-------------|---------------|
| 1 | Causal trace accuracy | Chain length ≥ 2, conflict detection, invalidation |
| 2 | Multi-timescale separation | Fast < 5 steps, slow > 10 steps, ordering correct |
| 3 | Geometry computation correctness | Signal bending > 0, mass amplification, no NaN |

In [ ]:
log.cell_start("Cell 7 — Test 1: Causal Trace Accuracy")

os.chdir(str(WORK_DIR / 'phases' / 'phase5'))
%run test_phase5_test1.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 7 — Test 1: Causal Trace Accuracy", "PASS")

In [ ]:
log.cell_start("Cell 8 — Test 2: Multi-Timescale Separation")

os.chdir(str(WORK_DIR / 'phases' / 'phase5'))
%run test_phase5_test2.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 8 — Test 2: Multi-Timescale Separation", "PASS")

In [ ]:
log.cell_start("Cell 9 — Test 3: Geometry Computation Correctness")

os.chdir(str(WORK_DIR / 'phases' / 'phase5'))
%run test_phase5_test3.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 9 — Test 3: Geometry Computation Correctness", "PASS")

---
## Cells 10–11: Demos

| Demo | Description |
|------|-------------|
| 1 | Trace why a conclusion was reached — penguin/bird causal reasoning with conflict detection |
| 2 | Fast vs slow node activation — timescale separation curves over 100 steps |

In [ ]:
log.cell_start("Cell 10 — Demo 1: Trace Why a Conclusion Was Reached")

os.chdir(str(WORK_DIR / 'phases' / 'phase5'))
%run demo_phase5_demo1.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 10 — Demo 1: Trace Why a Conclusion Was Reached", "PASS")

In [ ]:
log.cell_start("Cell 11 — Demo 2: Fast vs Slow Node Activation")

os.chdir(str(WORK_DIR / 'phases' / 'phase5'))
%run demo_phase5_demo2.py

from IPython.display import Image, display
img_path = WORK_DIR / 'phases' / 'phase5' / 'demo5_timescale_activation.png'
if img_path.exists():
    display(Image(filename=str(img_path), width=900))
    log.success("Timescale activation chart generated")
else:
    log.warning("Timescale chart not generated (matplotlib may be unavailable)")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 11 — Demo 2: Fast vs Slow Node Activation", "PASS")

---
## Cell 12: Interactive Exploration
Build custom causal graphs, process signals through CGN network, and explore the multi-timescale architecture.

In [ ]:
log.cell_start("Cell 12 — Interactive Exploration")

import torch
import torch.nn.functional as F
from cgn import CausalGeometryNode
from causal_graph import CausalGraph
from multi_timescale import MultiTimescaleCoordinator

# ── Build a custom causal knowledge base ──
print("  Building Custom Knowledge Graph:")
print("  " + "-" * 55)

cg_explore = CausalGraph()
custom_knowledge = [
    (0, 1, 1.0, "FLUX uses physics-inspired computation"),
    (1, 2, 0.9, "physics computation replaces weight matrices"),
    (2, 3, 0.85, "no weight matrices means no backprop needed"),
    (3, 4, 0.8, "no backprop means real-time learning"),
    (4, 5, 0.75, "real-time learning enables one-shot facts"),
    (0, 6, 0.9, "FLUX uses causal geometry nodes"),
    (6, 7, 0.85, "CGN stores WHY not just WHAT"),
    (7, 8, 0.8, "storing WHY enables explainability"),
]

node_names = {
    0: "FLUX", 1: "physics_compute", 2: "no_weights",
    3: "no_backprop", 4: "realtime_learn", 5: "oneshot_facts",
    6: "CGN", 7: "stores_WHY", 8: "explainable",
}

for src, tgt, weight, reason in custom_knowledge:
    cg_explore.add_arrow(src, tgt, weight=weight, reason=reason)
    print(f"    {node_names[src]:>16} → {node_names[tgt]:<16}  w={weight:.2f}")

# ── Trace reasoning chains ──
print("\n  Reasoning Traces:")
print("  " + "-" * 55)

for target, name in [(5, "oneshot_facts"), (8, "explainable")]:
    trace = cg_explore.trace_cause(target, depth=10)
    chain_str = " → ".join(node_names.get(n, str(n)) for n in trace.chain)
    print(f"    Why '{name}'?")
    print(f"      Chain: {chain_str}")
    print(f"      Total weight: {trace.total_weight:.2f}")
    log.metric(f"trace({name})", f"chain_len={len(trace.chain)}")

# ── Process text through full CGN pipeline ──
print("\n  Full Pipeline Processing:")
print("  " + "-" * 55)

explore_texts = [
    "Neural networks compute via matrix multiplication",
    "FLUX computes via geometric signal bending on manifolds",
    "Causal reasoning requires knowing WHY not just WHAT",
    "Temperature controls how much the system can change",
]

cgn_explore = MultiTimescaleCoordinator(512).to('cpu')
cse_cpu = cse.to('cpu')

for text in explore_texts:
    with torch.no_grad():
        wave = cse_cpu.encode(text)
        wave_vec = wave.full.mean(dim=0)
        feature = field.to('cpu').wave_to_feature(wave_vec).detach()
        cgn_explore.reset_states()
        result = cgn_explore.forward_with_traces(feature, steps=10)

    print(f"    '{text[:55]}'")
    print(f"      Output norm: {result.output.norm().item():.4f}  "
          f"Fast: {result.fast_activation:.4f}  "
          f"Slow: {result.slow_activation:.4f}  "
          f"Traces: {len(result.traces)}")

# ── CGN network statistics ──
print("\n  CGN Network Statistics:")
print("  " + "-" * 45)
stats = cgn_explore.stats()
for k, v in stats.items():
    if isinstance(v, float):
        print(f"    {k:<25}: {v:.4f}")
    elif isinstance(v, list):
        print(f"    {k:<25}: [{', '.join(f'{x:.3f}' for x in v)}]")
    else:
        print(f"    {k:<25}: {v}")
    log.metric(k, str(v))

# Restore to original device
cse = cse.to(DEVICE)
field = field.to(DEVICE)
del cgn_explore, cg_explore

print("\n  ✓ Interactive exploration complete")

log.cell_end("Cell 12 — Interactive Exploration", "PASS")

---
## Cell 13: View Results Summary & Full Log

In [ ]:
log.cell_start("Cell 13 — Results Summary & Log")

from IPython.display import Markdown, display

results_path = WORK_DIR / 'phases' / 'phase5' / 'RESULTS_PHASE_5.md'
if results_path.exists():
    display(Markdown(results_path.read_text()))
    log.success("Results displayed")
else:
    print("  ⚠ RESULTS_PHASE_5.md not yet generated — run tests first")
    print("  Tests auto-generate this file via PhaseResults utility.")
    log.warning("No RESULTS_PHASE_5.md found")

print("\n" + "=" * 60)
print("FULL PHASE 5 LOG")
print("=" * 60)
print(log.get_log_contents())

log.cell_end("Cell 13 — Results Summary & Log", "PASS")

---
## Cell 14: Final Upload — Logs to HuggingFace & GitHub
Pushes the complete Phase 5 log, results, and checkpoint to both platforms.

In [ ]:
log.cell_start("Cell 14 — Final Upload")

upload_logs_to_hf(phase=5, hf_token=HF_TOKEN)
log.success("Logs uploaded to HuggingFace Hub")

git_commit_and_push(
    files=[
        'logs/phase5.log',
        'results/',
        'phases/phase5/RESULTS_PHASE_5.md',
    ],
    message='Phase 5: Causal Geometry Nodes — training complete [auto-commit from Kaggle]',
    repo_dir=str(WORK_DIR),
)

log.cell_end("Cell 14 — Final Upload", "PASS")

print("\n" + "=" * 60)
print("✓ PHASE 5 COMPLETE")
print("=" * 60)
print(f"  Checkpoint: https://huggingface.co/UnseenGAP/FLUX")
print(f"  Logs:       https://huggingface.co/UnseenGAP/FLUX (logs/)")
print(f"  Code:       https://github.com/Unseengap/FLUX")
print(f"  Next:       Phase 6 — Three-Tier Memory System")
print("=" * 60)

---
## Cell 15: Save Artifacts to Kaggle Output

In [ ]:
log.cell_start("Cell 15 — Save Kaggle Artifacts")

import shutil

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

# ── Checkpoints ──
for fname in ['phase5.phase.pt']:
    src = WORK_DIR / 'checkpoints' / fname
    if src.exists():
        shutil.copy2(str(src), str(output_dir / src.name))
        print(f"  ✓ Checkpoint: {src.name} ({src.stat().st_size/1e6:.1f} MB)")

# ── Results and logs ──
for fpath in [
    WORK_DIR / 'phases' / 'phase5' / 'RESULTS_PHASE_5.md',
    WORK_DIR / 'logs' / 'phase5.log',
]:
    if fpath.exists():
        shutil.copy2(str(fpath), str(output_dir / fpath.name))
        print(f"  ✓ {fpath.name}")

# ── Demo images ──
for img in (WORK_DIR / 'phases' / 'phase5').glob('*.png'):
    shutil.copy2(str(img), str(output_dir / img.name))
    print(f"  ✓ {img.name}")

print(f"\n  Kaggle output artifacts:")
for f in sorted(output_dir.iterdir()):
    print(f"    {f.name} ({f.stat().st_size / 1e6:.2f} MB)")

log.cell_end("Cell 15 — Save Kaggle Artifacts", "PASS")

print("\n" + "=" * 60)
print("  PHASE 5: Causal Geometry Nodes — ALL DONE ✓")
print("=" * 60)

---
## Phase 5 Acceptance Criteria

| Criteria | Cell | Target |
|---|---|---|
| Every output has a traceable causal chain | Stage A / Test 1 | Chain length ≥ 2 |
| Fast nodes react in < 5 steps | Stage B / Test 2 | < 5 steps |
| Slow nodes react in > 10 steps | Stage B / Test 2 | > 10 steps |
| Geometry computation produces correct signal bending | Stage C / Test 3 | Bending > 0 |
| Causal invalidation works (disprove cause → invalidate conclusion) | Stage D / Test 1 | Affected > 0 |
| Mass amplifies output correctly | Stage C / Test 3 | High mass > low mass |
| All tests pass | Tests 1–3 | ✓ |
| Phase 1 checkpoint loads correctly | Cell 4 | ✓ |
| Phase 2 checkpoint loads correctly | Cell 4 | ✓ |
| Phase 4 checkpoint loads correctly | Cell 4 | ✓ |
| Phase 5 checkpoint saved | Cell 6 | `checkpoints/phase5.phase.pt` |
| Checkpoint uploaded to HuggingFace | Cell 6 | `UnseenGAP/FLUX` |
| Logs written to logs/phase5.log | All cells | ✓ |
| RESULTS_PHASE_5.md generated | Tests | ✓ |

### Key Design Decisions
- **No activation functions** — curvature bends signals, geometry IS the computation
- **Causal arrows** — every conclusion stores WHY it was reached (not just correlations)
- **Multi-timescale** — fast (τ=0.01, syntax) + medium (τ=0.1, semantics) + slow (τ=1.0, concepts)
- **Invalidation propagation** — disprove a cause → all dependent conclusions weakened
- **Mass = evidence** — more supporting evidence = stronger node influence
- **Manifold geometry** — Riemannian-inspired curvature, connection, and geodesic computation